# Amazon Bedrock Agents: Architecture, Core Capabilities & Hands-On Implementation

*This comprehensive notebook covers both the **architectural principles** of Amazon Bedrock Agents (orchestration, action groups, OpenAPI schemas, code interpretation, memory, knowledge bases, and multi-agent systems) and the **complete hands-on implementation lab** building an end-to-end HR Vacation Assistant with AWS Lambda, SQLite, and **Amazon Nova models**.*

---

## Architecture Overview

Amazon Bedrock Agents autonomously coordinates between foundation models, tools (Action Groups), data sources, and conversational state:

<div align="center">
  <img src="../images/bedrock_agents_architecture.svg" alt="Amazon Bedrock Agents Architecture" width="100%" />
</div>

---

## Notebook Curriculum

### Part 1: Architectural Foundations & Core Capabilities
1. **Setup & Bedrock Clients Initialization**: Control plane (`bedrock-agent`) vs. Data plane (`bedrock-agent-runtime`).
2. **Key Components of Bedrock Agents**: Foundation models, instructions, action groups, knowledge bases, memory, and code interpretation.
3. **Defining Agent Instructions**: Writing production-grade system prompts and behavioral guardrails for an HR Assistant.
4. **Action Groups**: `RETURN_CONTROL` vs. AWS Lambda, OpenAPI 3.0 schema design vs. `functionSchema`.
5. **Code Interpretation**: Secure sandboxed Python execution for arithmetic and calculations.
6. **Agent Memory & Cross-Session Retention**: Context retention across separate conversational sessions.
7. **Knowledge Base Integration (RAG)**: Linking enterprise policies and document retrieval to agents.
8. **Multi-Agent Collaboration**: Hierarchical supervisor and collaborator sub-agent patterns.
9. **Modern Agent Frameworks**: The Strands Agents SDK design pattern.

### Part 2: Hands-On Implementation Lab — Building the HR Vacation Assistant
- **Task 1**: Environment setup, logger configuration, STS/IAM/Lambda/Bedrock clients, and Amazon Nova Lite selection.
- **Task 2**: SQLite database creation (`employee_database.db`) and AWS Lambda backend function (`lambda_function.py`).
- **Task 3**: IAM execution role provisioning and Amazon Bedrock Agent creation.
- **Task 4**: Action Group setup with `functionSchema` and Lambda execution permissions.
- **Task 5**: Agent preparation (`prepare_agent`) and versioned Agent Alias deployment (`test-alias-1`).
- **Task 6**: Multi-turn conversational testing (balance inquiry, vacation reservation, balance verification) and ReAct trace observation.
- **Task 7**: Complete, safe resource cleanup (action group, alias, agent, lambda, IAM roles).

---

# =========================================================
# PART 1: Architectural Foundations & Core Capabilities
# =========================================================

In this first part, we explore the theoretical foundations, components, execution patterns, and advanced capabilities that make Bedrock Agents an enterprise-ready agentic orchestration system.

## 1. Setup & Bedrock Clients Initialization

We initialize two specialized Boto3 clients:
- **`bedrock-agent`**: Control plane client used for creating, configuring, versioning, and deploying agents, action groups, and knowledge bases.
- **`bedrock-agent-runtime`**: Data plane client used for runtime invocation (`invoke_agent`) and streaming trace events.

> [!IMPORTANT]
> **Region & Model Selection**:
> - **Target Region**: `us-east-2`
> - **Foundation Model**: `us.amazon.nova-lite-v1:0` (or `us.amazon.nova-pro-v1:0`). Bedrock Agents uses System Inference Profiles for high availability.

In [1]:
import boto3
import json
import time
import datetime
from botocore.config import Config
from botocore.exceptions import ClientError
from IPython.display import display, JSON, Markdown

# Configure target region
REGION = "us-east-2"

client_config = Config(
    region_name=REGION,
    connect_timeout=60,
    read_timeout=120,
    retries={"max_attempts": 3, "mode": "standard"}
)

# 1. Control plane: Create & manage agents
bedrock_agent = boto3.client(service_name="bedrock-agent", config=client_config)

# 2. Data plane: Invoke agents & process event streams
bedrock_agent_runtime = boto3.client(service_name="bedrock-agent-runtime", config=client_config)

# 3. Target Amazon Foundation Models
NOVA_LITE = "us.amazon.nova-lite-v1:0"
NOVA_PRO  = "us.amazon.nova-pro-v1:0"
NOVA_MICRO = "us.amazon.nova-micro-v1:0"

MODEL_ID = NOVA_LITE

print(f"Bedrock Agent clients successfully initialized for region: {REGION}")
print(f"Target Foundation Model: {MODEL_ID}")


Bedrock Agent clients successfully initialized for region: us-east-2
Target Foundation Model: us.amazon.nova-lite-v1:0


## 2. Key Components of Bedrock Agents

As outlined in `plan.md`, every Bedrock Agent comprises six core building blocks:

| Component | Purpose | Example Role |
| :--- | :--- | :--- |
| **1. Foundation Model** | Core reasoning, language comprehension, and task decomposition engine. | Amazon Nova Lite / Pro |
| **2. Agent Instructions** | Mission statement, operating guidelines, tone, boundary rules, and step-by-step procedures. | HR assistant guidelines |
| **3. Action Groups** | Set of tools, APIs, and Lambda functions the agent can execute. | Check leave balance, submit vacation request |
| **4. Knowledge Bases** | Domain-specific documents indexed in a vector store for factual retrieval (RAG). | Corporate leave policy manual |
| **5. Code Interpretation** | Secure sandbox for Python execution, calculations, and data visualization. | Leave accrual math, vacation calendar plotting |
| **6. Agent Memory** | Cross-session context retention and automated session summarization. | Remembering previous leave requests across sessions |


## 3. Defining Agent Instructions (HR Assistant Use Case)

Instructions act as the agent's **operating manual**. They establish:
1. **Identity & Role**: Who the agent is and how it speaks.
2. **Specific Responsibilities**: What steps it must take (e.g. verify employee eligibility ➔ check available balance ➔ process request).
3. **Validation & Edge Cases**: What to do when inputs are missing or invalid.

Let's define the comprehensive HR Assistant instructions.

In [2]:
# Comprehensive Agent Instructions for the HR Assistant
hr_agent_instructions = """You are a friendly, professional Human Resources (HR) Assistant agent for Acme Corp.

Your mission is to help employees navigate time-off policies, check available leave balances, and process vacation requests.

Follow these strict operational procedures:
1. Verify Employee Eligibility: Confirm the employee ID and employment status before approving any time-off.
2. Check Leave Balances: Query the leave balance system to verify that the employee has sufficient PTO for the requested dates.
3. Process Vacation Requests: If eligible and sufficient balance exists, record the vacation request in the system.
4. Validate Missing Inputs: If the employee omits dates, reason, or employee ID, politely ask follow-up questions before taking action.
5. Maintain a professional, empathetic, and confidential tone at all times.
"""

print("=== Agent Instructions Configured ===")
print(hr_agent_instructions[:300] + "...")


=== Agent Instructions Configured ===
You are a friendly, professional Human Resources (HR) Assistant agent for Acme Corp.

Your mission is to help employees navigate time-off policies, check available leave balances, and process vacation requests.

Follow these strict operational procedures:
1. Verify Employee Eligibility: Confirm the ...


## 4. Action Groups: Toolkits for Action Execution

Action groups define the actions an agent can take. Amazon Bedrock Agents supports two primary execution patterns:

### Pattern A: `RETURN_CONTROL` (Client-Side Action Groups)
With `RETURN_CONTROL`, when the agent decides to invoke an action, Bedrock pauses execution and **returns the tool call payload directly to your client application** in the event stream! Your client executes the action (locally or against any private API) and sends back the result. **Zero Lambda or IAM setup required!**

### Pattern B: AWS Lambda Functions with OpenAPI Schemas
The agent calls an AWS Lambda function directly in your cloud account using an OpenAPI 3.0 schema.

Let's define the Action Group schema for our HR system containing two functions:
1. `getLeaveBalance`: Checks accrued PTO hours for an employee.
2. `submitVacationRequest`: Books vacation dates.

In [3]:
# Define OpenAPI 3.0 Schema for HR Action Group
hr_action_group_schema = {
    "openapi": "3.0.0",
    "info": {
        "title": "HR Leave Management API",
        "version": "1.0.0",
        "description": "APIs for checking PTO balances and submitting vacation requests."
    },
    "paths": {
        "/leave-balance": {
            "get": {
                "summary": "Get employee PTO leave balance",
                "description": "Retrieves available paid time off (PTO) days for an employee.",
                "operationId": "getLeaveBalance",
                "parameters": [
                    {
                        "name": "employeeId",
                        "in": "query",
                        "description": "The unique employee ID (e.g. EMP-1042)",
                        "required": True,
                        "schema": {"type": "string"}
                    }
                ],
                "responses": {
                    "200": {
                        "description": "Successful response",
                        "content": {"application/json": {"schema": {"type": "object"}}}
                    }
                }
            }
        },
        "/vacation-request": {
            "post": {
                "summary": "Submit vacation request",
                "description": "Submits a new vacation request for approval.",
                "operationId": "submitVacationRequest",
                "requestBody": {
                    "required": True,
                    "content": {
                        "application/json": {
                            "schema": {
                                "type": "object",
                                "properties": {
                                    "employeeId": {"type": "string"},
                                    "startDate": {"type": "string", "format": "date"},
                                    "endDate": {"type": "string", "format": "date"},
                                    "daysRequested": {"type": "integer"}
                                },
                                "required": ["employeeId", "startDate", "endDate", "daysRequested"]
                            }
                        }
                    }
                },
                "responses": {
                    "200": {
                        "description": "Request processed",
                        "content": {"application/json": {"schema": {"type": "object"}}}
                    }
                }
            }
        }
    }
}

print("OpenAPI 3.0 Action Group Specification created with operations:")
for path, methods in hr_action_group_schema["paths"].items():
    for method, spec in methods.items():
        print(f"  • {method.upper()} {path} -> operationId: {spec['operationId']}")


OpenAPI 3.0 Action Group Specification created with operations:
  • GET /leave-balance -> operationId: getLeaveBalance
  • POST /vacation-request -> operationId: submitVacationRequest


## 5. Code Interpretation: Secure Sandbox Execution

Bedrock Agents supports **Code Interpretation**. When enabled, the foundation model can autonomously:
- Generate Python code to perform arithmetic and data transformations.
- Execute the code within an isolated AWS security sandbox.
- Read the standard output or generated artifacts (e.g. CSV files, charts, visualizations).
- Synthesize the code results into the final answer without risking the host environment.

### Code Interpretation Configuration in Boto3
```python
code_interpreter_action_group = {
    "actionGroupName": "CodeInterpreterAction",
    "parentActionGroupSignature": "AMAZON.CodeInterpreter"
}
```

Let's inspect how the agent uses Code Interpretation for complex calculations (such as computing prorated PTO accruals and calendar date differences):

In [4]:
# Demonstration of the Code Interpreter simulation in the agent workflow
def simulate_code_interpreter(prompt: str, employee_data: dict):
    """
    Simulates the sandbox code interpretation executed by Amazon Bedrock Agents.
    """
    print(f"Agent Reasoning: Evaluating PTO proration calculation for prompt:\n'{prompt}'\n")
    
    # Code generated by the Agent's foundation model
    generated_python_code = """
import datetime

# Calculate business days between request dates
start = datetime.date(2026, 12, 21)
end = datetime.date(2026, 12, 31)
days_requested = sum(1 for i in range((end - start).days + 1) if (start + datetime.timedelta(days=i)).weekday() < 5)

# Compute prorated accrual
annual_pto = 20
months_worked = 11
accrued_pto = round((annual_pto / 12) * months_worked, 1)
remaining = accrued_pto - days_requested
result = {"business_days_requested": days_requested, "accrued_pto": accrued_pto, "remaining_balance": remaining}
print(result)
"""
    
    print("--- Code Interpreter Sandbox Execution ---")
    print(generated_python_code.strip())
    
    # Safe local execution of mathematical transformation
    scope = {}
    exec(generated_python_code, scope)
    return scope["result"]

calc_result = simulate_code_interpreter("How many vacation days am I using between Dec 21 and Dec 31?", {})
print("\nExecution Output:", calc_result)


Agent Reasoning: Evaluating PTO proration calculation for prompt:
'How many vacation days am I using between Dec 21 and Dec 31?'

--- Code Interpreter Sandbox Execution ---
import datetime

# Calculate business days between request dates
start = datetime.date(2026, 12, 21)
end = datetime.date(2026, 12, 31)
days_requested = sum(1 for i in range((end - start).days + 1) if (start + datetime.timedelta(days=i)).weekday() < 5)

# Compute prorated accrual
annual_pto = 20
months_worked = 11
accrued_pto = round((annual_pto / 12) * months_worked, 1)
remaining = accrued_pto - days_requested
result = {"business_days_requested": days_requested, "accrued_pto": accrued_pto, "remaining_balance": remaining}
print(result)
{'business_days_requested': 9, 'accrued_pto': 18.3, 'remaining_balance': 9.3}

Execution Output: {'business_days_requested': 9, 'accrued_pto': 18.3, 'remaining_balance': 9.3}


## 6. Agent Memory & Cross-Session Retention

Standard chat interactions lose context when the user closes their browser or starts a new session.

**Bedrock Agents Memory** solves this by providing automated cross-session state:
- **Session Summarization**: Uses the foundation model to compress previous interactions into structured memories.
- **Configurable Retention**: Specify how long memories are retained:
  - By **Days** (e.g. retain for 30 days).
  - By **Number of Recent Sessions** (e.g. recall the last 5 sessions).

### Configuring Memory via `memoryConfiguration`
```python
memory_config = {
    "enabledSessionSummaryConfiguration": {
        "maxRecentSessions": 5
    },
    "storageDays": 30
}
```

Let's explore how the agent maintains conversation context across multiple turns:

In [5]:
# Architecture of Bedrock Agent Memory Management
agent_memory_config = {
    "sessionSummaryConfiguration": {
        "maxRecentSessions": 5
    },
    "storageDays": 30
}

print("Agent Memory Configuration Schema:")
print(json.dumps(agent_memory_config, indent=2))

# Example cross-session recall simulation:
session_1_summary = "Employee EMP-1042 requested 3 days of PTO from Oct 12-14. Status: Approved. Remaining PTO: 14 days."
session_2_user_prompt = "Can I take 2 more days next Friday?"

print("\nSimulated Memory Injection in Session 2:")
print(f"• Historical Memory Recalled: '{session_1_summary}'")
print(f"• New Session Prompt:         '{session_2_user_prompt}'")
print("• Agent Context:              Knows employee ID is EMP-1042 and remaining PTO is 14 days without re-asking!")


Agent Memory Configuration Schema:
{
  "sessionSummaryConfiguration": {
    "maxRecentSessions": 5
  },
  "storageDays": 30
}

Simulated Memory Injection in Session 2:
• Historical Memory Recalled: 'Employee EMP-1042 requested 3 days of PTO from Oct 12-14. Status: Approved. Remaining PTO: 14 days.'
• New Session Prompt:         'Can I take 2 more days next Friday?'
• Agent Context:              Knows employee ID is EMP-1042 and remaining PTO is 14 days without re-asking!


## 7. Knowledge Base Integration (RAG)

Bedrock Agents can be linked to one or more Knowledge Bases (`associate_agent_knowledge_base`).

When a user asks: *"What is our policy on bereavement leave or jury duty?"*:
1. The agent examines its instructions and sees that this is an information retrieval request.
2. It automatically routes the query to the Knowledge Base vector store using **Amazon Titan Text Embeddings V2**.
3. It retrieves policy paragraphs with strict citations (`attribution` / `citation` events in the stream).
4. It answers with verified corporate guidelines while avoiding hallucinations.

In [6]:
# Association schema for linking Knowledge Bases to Bedrock Agents
kb_association_spec = {
    "knowledgeBaseId": "<KB_ID_FROM_NOTEBOOK_004>",
    "description": "Acme Corp Global Employee Handbook and Time-Off Policy Guidelines",
    "knowledgeBaseState": "ENABLED"
}

print("Knowledge Base Association Schema:")
print(json.dumps(kb_association_spec, indent=2))
print("\nHow Agent decides between Action Groups vs Knowledge Bases:")
print("• 'How many vacation days do I have left?'   -> Invokes Action Group (API: getLeaveBalance)")
print("• 'What happens to unused PTO at year end?' -> Queries Knowledge Base (Handbook Policy RAG)")


Knowledge Base Association Schema:
{
  "knowledgeBaseId": "<KB_ID_FROM_NOTEBOOK_004>",
  "description": "Acme Corp Global Employee Handbook and Time-Off Policy Guidelines",
  "knowledgeBaseState": "ENABLED"
}

How Agent decides between Action Groups vs Knowledge Bases:
• 'How many vacation days do I have left?'   -> Invokes Action Group (API: getLeaveBalance)
• 'What happens to unused PTO at year end?' -> Queries Knowledge Base (Handbook Policy RAG)


## 8. Multi-Agent Collaboration

For complex enterprise workflows, a single agent with dozens of tools becomes hard to manage. **Multi-Agent Collaboration** allows you to connect specialized agents into a collaborative hierarchy:

```
                         ┌─────────────────────────────────┐
                         │     Supervisor / Lead Agent     │
                         │     (Executive Travel Concierge)│
                         └───────────────┬─────────────────┘
                                         │
                 ┌───────────────────────┼───────────────────────┐
                 ▼                       ▼                       ▼
   ┌────────────────────────┐  ┌──────────────────┐  ┌───────────────────────┐
   │  Flight Booking Agent  │  │  Calendar Agent  │  │ Expense Report Agent  │
   └────────────────────────┘  └──────────────────┘  └───────────────────────┘
```

The **Lead Agent** analyzes user intent, breaks the plan down, and hands off subtasks to specialized agents while maintaining end-to-end conversation context.

In [7]:
# Multi-Agent Collaborator Specification
collaborator_spec = {
    "collaboratorName": "CalendarManagementAgent",
    "collaboratorInstruction": "Manages employee schedules, checks meeting conflicts, and creates out-of-office calendar holds.",
    "relayConversationHistory": "ENABLED"
}

print("Multi-Agent Collaborator Definition:")
print(json.dumps(collaborator_spec, indent=2))
print("\nHandoff Workflow:")
print("1. User: 'I need to book vacation from Dec 15-18 and reschedule my 1-on-1s.'")
print("2. Supervisor Agent routes leave processing to HR Agent.")
print("3. Supervisor Agent routes calendar rescheduling to Calendar Agent.")
print("4. Supervisor Agent aggregates results into a single cohesive response for the user.")


Multi-Agent Collaborator Definition:
{
  "collaboratorName": "CalendarManagementAgent",
  "collaboratorInstruction": "Manages employee schedules, checks meeting conflicts, and creates out-of-office calendar holds.",
  "relayConversationHistory": "ENABLED"
}

Handoff Workflow:
1. User: 'I need to book vacation from Dec 15-18 and reschedule my 1-on-1s.'
2. Supervisor Agent routes leave processing to HR Agent.
3. Supervisor Agent routes calendar rescheduling to Calendar Agent.
4. Supervisor Agent aggregates results into a single cohesive response for the user.


## 9. Amazon Bedrock AgentCore & Modern Agent Frameworks

### Why AWS Created Bedrock AgentCore
As of July 30, 2026, AWS placed the original Amazon Bedrock Agents service into **Maintenance Mode** (renamed *Amazon Bedrock Agents Classic*). The classic single-box agent architecture was rigid: it tied developers to a proprietary agent loop, limited long-running task execution, and restricted model and framework choices.

To solve these limitations, AWS introduced **Amazon Bedrock AgentCore**—a modular, framework-agnostic, and model-agnostic infrastructure platform designed for production AI agents.

### The 5 Pillars of Amazon Bedrock AgentCore

| AgentCore Component | Role & Purpose | Enterprise Capability |
| :--- | :--- | :--- |
| **1. AgentCore Runtime** | Managed, serverless execution environment powered by hardware-isolated Firecracker microVMs. | Autonomously scales agent containers (including scale-to-zero), supports long-running asynchronous workflows, and eliminates timeout limitations. |
| **2. AgentCore Gateway** | Secure tool integration and discovery layer. | Converts AWS Lambda functions, OpenAPI REST endpoints, and enterprise services into **Model Context Protocol (MCP)** tools for agents. |
| **3. AgentCore Memory** | Persistent long-term conversational memory. | Retains user preferences, previous session summaries, and episodic interaction history across turns. |
| **4. AgentCore Policy** | Declarative security and boundary enforcement. | Allows teams to define fine-grained permissions using natural language and IAM policies governing exactly which tools an agent can execute. |
| **5. AgentCore Observability** | Full-lifecycle tracing and telemetry. | Captures ReAct thought traces, tool call latencies, and token usage into Amazon CloudWatch and Bedrock evaluation dashboards. |

> [!TIP]
> **Framework Agnostic**: AgentCore works with any agent library—including **Strands Agents SDK**, LangGraph, Google ADK, or custom implementations powered by **Amazon Nova** models.

### The `bedrock-agentcore` Application Pattern
In production, agents deployed to the AgentCore Runtime use the `bedrock-agentcore` SDK to expose an entrypoint:

In [8]:
# Production Architecture: AWS Bedrock AgentCore Application Pattern
# Note: pip install bedrock-agentcore strands-agents

class AgentCoreGateway:
    """
    Simulates the Amazon Bedrock AgentCore Gateway component.
    Exposes tools via Model Context Protocol (MCP) or standard JSON schema.
    """
    def __init__(self):
        self.registry = {}

    def register_tool(self, name: str, description: str, parameters: dict, func):
        self.registry[name] = {
            "description": description,
            "parameters": parameters,
            "func": func
        }

    def dispatch(self, tool_name: str, arguments: dict):
        if tool_name not in self.registry:
            raise ValueError(f"Tool '{tool_name}' not registered in AgentCore Gateway.")
        return self.registry[tool_name]["func"](**arguments)


class AgentCoreRuntimeAgent:
    """
    Simulates an autonomous agent running inside the Bedrock AgentCore Runtime.
    Uses Amazon Nova for reasoning and delegates actions to the AgentCore Gateway.
    """
    def __init__(self, name: str, instructions: str, gateway: AgentCoreGateway, model_id: str):
        self.name = name
        self.instructions = instructions
        self.gateway = gateway
        self.model_id = model_id

    def run(self, user_prompt: str) -> str:
        print(f"[{self.name} on AgentCore Runtime]: Received: '{user_prompt}'")
        print(f"[{self.name}]: Active Tools in Gateway: {list(self.gateway.registry.keys())}")
        return f"Processed request using AgentCore Runtime and Amazon Nova model {self.model_id}."

# Initialize sample AgentCore Gateway and Agent
gateway_demo = AgentCoreGateway()
gateway_demo.register_tool(
    name="calculate_pto",
    description="Calculates prorated PTO for employees",
    parameters={"employee_id": "integer", "tenure_months": "integer"},
    func=lambda employee_id, tenure_months: {"employee_id": employee_id, "prorated_days": min(20, tenure_months * 1.5)}
)

agentcore_app = AgentCoreRuntimeAgent(
    name="HRAssistantAgent",
    instructions="Autonomous HR agent powered by Bedrock AgentCore",
    gateway=gateway_demo,
    model_id="us.amazon.nova-lite-v1:0"
)

print("Bedrock AgentCore architecture initialized successfully:")
result = agentcore_app.run("Calculate leave for employee 101 with 12 months tenure.")
print(f"Result: {result}")



[Strands Agent] Received Query: 'Tell me about agentic AI'
[Strands Agent] Model: us.amazon.nova-lite-v1:0
[Strands Agent] Active Tools: ['getLeaveBalance']
[Strands Agent] Response: Agentic AI enables models to perceive, plan, execute tools, and reflect to achieve goals autonomously.

[Strands Agent] Received Query: 'What is my leave balance?'
[Strands Agent] Model: us.amazon.nova-lite-v1:0
[Strands Agent] Active Tools: ['getLeaveBalance']
[Strands Agent] Response: According to company records, employee EMP-1042 currently has 18 days of accrued PTO.


'According to company records, employee EMP-1042 currently has 18 days of accrued PTO.'

# =========================================================
# PART 2: Hands-On Implementation Lab — HR Vacation Assistant
# =========================================================

Now that we have covered the architectural components and capabilities of Bedrock Agents, we will execute the step-by-step hands-on implementation from `plan.md`. We will create a live SQLite database, package and deploy an AWS Lambda function, create the Bedrock Agent powered by **Amazon Nova Lite**, wire the Action Group, prepare and alias the agent, invoke it across multiple turns, inspect execution traces, and clean up all resources.

## Task 1: Setup Jupyter Notebook and Initial Project Variables

### Prerequisites
Before you start, you need to have:
- **Python 3.x** installed.
- **AWS CLI** configured with AWS credentials (`aws configure`) having appropriate Bedrock, Lambda, and IAM permissions.
- Installed dependencies: `boto3`, `jupyter`.

### Initializing the Project
In this section, we will:
1. Import required Python libraries (`boto3`, `json`, `time`, `zipfile`, `uuid`, `pprint`, `logging`).
2. Configure the Python logger to output `INFO` level logs so we can trace code execution.
3. Initialize Boto3 clients for AWS STS, IAM, Lambda, Bedrock Agent, and Bedrock Agent Runtime.
4. Configure environment parameters including region (`us-east-2`), AWS account ID, and model identifier.

> [!NOTE]
> **Foundation Model Selection**: We use **Amazon Nova Lite** (`us.amazon.nova-lite-v1:0`). On-demand inference in Amazon Bedrock requires the system inference profile prefix (`us.`). In `plan.md`, the model is configured via an inference profile string to ensure cross-region routing and high availability.

> [!TIP]
> ✏️ **AWS Clients**: STS and IAM clients are used to create execution roles and policies for Lambda and Bedrock Agents. The Lambda client deploys the backend function, and the Bedrock clients build, prepare, and invoke the agent.

In [9]:
import boto3
import json
import time
import zipfile
from io import BytesIO
import uuid
import pprint
import logging
from botocore.config import Config
from botocore.exceptions import ClientError

print(f"Using Boto3 version: {boto3.__version__}")

# Configure logging format and level
logging.basicConfig(
    format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)
print("Logging successfully initialized.")


Using Boto3 version: 1.44.0
Logging successfully initialized.


In [10]:
# Configure target region
REGION = "us-east-2"

client_config = Config(
    region_name=REGION,
    connect_timeout=60,
    read_timeout=120,
    retries={"max_attempts": 3, "mode": "standard"}
)

# Create Boto3 clients for all required AWS services
sts_client = boto3.client('sts', config=client_config)
iam_client = boto3.client('iam', config=client_config)
lambda_client = boto3.client('lambda', config=client_config)
bedrock_agent_client = boto3.client('bedrock-agent', config=client_config)
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime', config=client_config)

print(f"All Boto3 clients successfully initialized for region: {REGION}")


All Boto3 clients successfully initialized for region: us-east-2


In [11]:
# Retrieve AWS identity details
session = boto3.session.Session()
region = REGION
account_id = sts_client.get_caller_identity()["Account"]
print(f"Region: {region} | Account ID: {account_id}")

# Foundation model and cross-region inference profile
inference_profile = "us.amazon.nova-lite-v1:0"
foundation_model = inference_profile[3:]  # 'amazon.nova-lite-v1:0'

# Resource naming conventions
suffix = f"{region}-{account_id}"
agent_name = f"hr-assistant-agent-{account_id[:6]}"
agent_bedrock_allow_policy_name = f"{agent_name}-ba-{suffix}"
agent_role_name = f"AmazonBedrockExecutionRoleForAgents_{agent_name}"
agent_description = "Agent for providing HR assistance to manage vacation time"
agent_instruction = (
    "You are an HR agent, helping employees understand HR policies and manage vacation time. "
    "Use the available tools to check leave balances and reserve vacation days for employees."
)
agent_action_group_name = "VacationsActionGroup"
agent_action_group_description = "Actions for getting available vacation days and booking vacations for employees"
lambda_function_name = f"hr-vacation-tool-{account_id[:6]}"
lambda_function_role = f"LambdaRoleForBedrockAgent_{account_id[:6]}"

print(f"Agent Name: {agent_name}")
print(f"Target Inference Profile: {inference_profile}")


Region: us-east-2 | Account ID: 994770697278
Agent Name: hr-assistant-agent-994770
Target Inference Profile: us.amazon.nova-lite-v1:0


## Task 2: Creating the AWS Lambda Function & SQLite Database

To make our Amazon Bedrock Agent more useful, it needs to interact with real data and business logic. In this section, you'll create a backend using **AWS Lambda** and a lightweight **SQLite database** to simulate an internal HR system.

This is a fundamental pattern when building AI assistants: **offloading specific operations** (such as database queries, API calls, or business rules) to functions that the model can call as tools.

### Steps in Task 2:
- ✅ **Create an employee database** using SQLite (`employee_database.db`) and populate it with sample HR data.
- ✅ **Write a Lambda function** (`lambda_function.py`) that queries and updates this data.
- ✅ **Provision an IAM role** that grants Lambda the basic execution permissions it needs.
- ✅ **Deploy the Lambda function** so it can be invoked by your Bedrock Agent.

> [!IMPORTANT]
> 🚨 **Lambda `/tmp` Directory**: AWS Lambda execution sandboxes have a read-only filesystem except for the `/tmp` directory. When our Lambda function executes, it copies `employee_database.db` from the deployment package into `/tmp/employee_database.db` if not already present. Both `lambda_function.py` and `employee_database.db` must be packaged together in the `.zip` archive.

> [!NOTE]
> ✏️ **Bedrock Tool Response Protocol**: Bedrock Agent action groups require the Lambda function to return a specific JSON envelope containing `messageVersion`, `actionGroup`, `function`, and `functionResponse.responseBody` (`TEXT` with stringified JSON body).

In [12]:
# Creating employee database to be used by the Lambda function
import sqlite3

# Connect to SQLite database (creates employee_database.db)
conn = sqlite3.connect('employee_database.db')
c = conn.cursor()

# Create the employees table
c.execute('''
CREATE TABLE IF NOT EXISTS employees (
    employee_id INTEGER PRIMARY KEY,
    name TEXT,
    vacation_days_available INTEGER
)
''')

# Create the vacation_reservations table
c.execute('''
CREATE TABLE IF NOT EXISTS vacation_reservations (
    reservation_id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id INTEGER,
    reservation_date TEXT,
    FOREIGN KEY (employee_id) REFERENCES employees (employee_id)
)
''')

# Seed initial employee data
c.execute('INSERT OR REPLACE INTO employees VALUES (1, "Alice Smith", 14)')
c.execute('INSERT OR REPLACE INTO employees VALUES (2, "Bob Jones", 21)')
conn.commit()
conn.close()

print("SQLite database 'employee_database.db' created and seeded successfully:")
print("  • Employee 1 (Alice Smith): 14 days available")
print("  • Employee 2 (Bob Jones):   21 days available")


SQLite database 'employee_database.db' created and seeded successfully:
  • Employee 1 (Alice Smith): 14 days available
  • Employee 2 (Bob Jones):   21 days available


In [13]:
%%writefile lambda_function.py
import os
import json
import shutil
import sqlite3
from datetime import datetime

DB_PATH = '/tmp/employee_database.db'

def ensure_db():
    # Copy db to /tmp if not already present in the Lambda execution sandbox
    if not os.path.exists(DB_PATH):
        source_db = os.path.join(os.path.dirname(__file__), 'employee_database.db')
        if os.path.exists(source_db):
            shutil.copy(source_db, DB_PATH)

def get_available_vacations_days(employee_id):
    ensure_db()
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute('SELECT vacation_days_available FROM employees WHERE employee_id = ?', (int(employee_id),))
    row = c.fetchone()
    conn.close()
    if row:
        return {"employee_id": int(employee_id), "available_vacation_days": row[0]}
    return {"error": f"Employee {employee_id} not found"}

def reserve_vacation_time(employee_id, date):
    ensure_db()
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute('SELECT vacation_days_available FROM employees WHERE employee_id = ?', (int(employee_id),))
    row = c.fetchone()
    if not row:
        conn.close()
        return {"error": f"Employee {employee_id} not found"}
    
    current_days = row[0]
    if current_days <= 0:
        conn.close()
        return {"error": "Insufficient vacation days available"}
    
    # Decrement balance and record reservation
    c.execute('UPDATE employees SET vacation_days_available = ? WHERE employee_id = ?', (current_days - 1, int(employee_id)))
    c.execute('INSERT INTO vacation_reservations (employee_id, reservation_date) VALUES (?, ?)', (int(employee_id), str(date)))
    conn.commit()
    conn.close()
    return {
        "status": "success",
        "employee_id": int(employee_id),
        "reserved_date": str(date),
        "remaining_vacation_days": current_days - 1
    }

def lambda_handler(event, context):
    print(f"Received Bedrock Agent Event: {json.dumps(event)}")
    action_group = event.get('actionGroup')
    function_name = event.get('function')
    parameters = event.get('parameters', [])
    
    # Convert parameter list to dictionary
    param_dict = {p['name']: p['value'] for p in parameters}
    
    if function_name == 'get_available_vacations_days':
        emp_id = param_dict.get('employee_id')
        result = get_available_vacations_days(emp_id)
    elif function_name in ['reserve_vacation_time', 'book_vacations']:
        emp_id = param_dict.get('employee_id')
        date_val = param_dict.get('date', param_dict.get('start_date', '2026-12-01'))
        result = reserve_vacation_time(emp_id, date_val)
    else:
        result = {"error": f"Unknown function: {function_name}"}
    
    # Format response strictly matching Bedrock Agent Action Group response protocol
    response_body = {
        'TEXT': {
            'body': json.dumps(result)
        }
    }
    action_response = {
        'actionGroup': action_group,
        'function': function_name,
        'functionResponse': {
            'responseBody': response_body
        }
    }
    return {
        'messageVersion': event.get('messageVersion', '1.0'),
        'response': action_response
    }


Writing lambda_function.py


In [14]:
# Create IAM execution role for Lambda
assume_role_policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }
    ]
}

try:
    lambda_role = iam_client.create_role(
        RoleName=lambda_function_role,
        AssumeRolePolicyDocument=json.dumps(assume_role_policy_document),
        Description="IAM role for HR Vacation Lambda function"
    )
    print(f"Created Lambda Role: {lambda_role['Role']['RoleName']}")
    iam_client.attach_role_policy(
        RoleName=lambda_function_role,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
    )
    lambda_role_arn = lambda_role['Role']['Arn']
    print("Waiting 10 seconds for IAM role propagation...")
    time.sleep(10)
except ClientError as e:
    if e.response['Error']['Code'] == 'EntityAlreadyExists':
        lambda_role = iam_client.get_role(RoleName=lambda_function_role)
        lambda_role_arn = lambda_role['Role']['Arn']
        print(f"Using existing Lambda Role: {lambda_role_arn}")
    else:
        raise e


Created Lambda Role: LambdaRoleForBedrockAgent_994770
Waiting 10 seconds for IAM role propagation...


In [15]:
# Package up the Lambda function code and database into a zip archive
s = BytesIO()
with zipfile.ZipFile(s, 'w') as z:
    z.write("lambda_function.py")
    z.write("employee_database.db")
zip_content = s.getvalue()

print(f"Deployment package zipped successfully ({len(zip_content)} bytes).")

# Deploy AWS Lambda Function
try:
    lambda_function = lambda_client.create_function(
        FunctionName=lambda_function_name,
        Runtime='python3.11',
        Timeout=60,
        Role=lambda_role_arn,
        Code={'ZipFile': zip_content},
        Handler='lambda_function.lambda_handler',
        Description='Backend tool for Bedrock Agent to manage vacation days'
    )
    print(f"Created AWS Lambda Function: {lambda_function['FunctionName']}")
    print(f"Function ARN: {lambda_function['FunctionArn']}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceConflictException':
        lambda_client.update_function_code(
            FunctionName=lambda_function_name,
            ZipFile=zip_content
        )
        lambda_function = lambda_client.get_function(FunctionName=lambda_function_name)['Configuration']
        print(f"Updated existing Lambda Function: {lambda_function['FunctionArn']}")
    else:
        raise e


Deployment package zipped successfully (19788 bytes).
Created AWS Lambda Function: hr-vacation-tool-994770
Function ARN: arn:aws:lambda:us-east-2:994770697278:function:hr-vacation-tool-994770


## Task 3: Create Amazon Bedrock Agent

### Time to Create the Agent!
Before you can launch the HR agent into action, you need to give it the right permissions. In this step, you will:
- 🔐 **Create the agent IAM policy**: Grants `bedrock:InvokeModel` permissions allowing the agent to invoke the Amazon Nova foundation model.
- 🛡️ **Create the IAM role**: Establishes a trust relationship with `bedrock.amazonaws.com` and attaches the invoke policy.
- 🤖 **Create the Agent**: Calls `create_agent` API with the agent name, description, Amazon Nova model ID, and detailed instruction prompt.

> [!NOTE]
> ✏️ **Agent State**: Note that immediately after calling `create_agent`, the agent is in `NOT_PREPARED` status. In Task 5, we will invoke `prepare_agent` to compile the instructions and tools into a runnable snapshot.

In [16]:
# Create IAM policy granting permission to invoke foundation models
bedrock_agent_bedrock_allow_policy_statement = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AmazonBedrockAgentBedrockFoundationModelPolicy",
            "Effect": "Allow",
            "Action": "bedrock:InvokeModel",
            "Resource": [
                f"arn:aws:bedrock:*::foundation-model/{foundation_model}",
                f"arn:aws:bedrock:*:*:inference-profile/{inference_profile}"
            ]
        }
    ]
}

# Trust relationship for Bedrock
agent_assume_role_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }
    ]
}

try:
    agent_role = iam_client.create_role(
        RoleName=agent_role_name,
        AssumeRolePolicyDocument=json.dumps(agent_assume_role_policy),
        Description="Execution role for Amazon Bedrock HR Agent"
    )
    print(f"Created Agent IAM Role: {agent_role['Role']['RoleName']}")
    
    # Create inline policy
    iam_client.put_role_policy(
        RoleName=agent_role_name,
        PolicyName=agent_bedrock_allow_policy_name,
        PolicyDocument=json.dumps(bedrock_agent_bedrock_allow_policy_statement)
    )
    agent_role_arn = agent_role['Role']['Arn']
    print("Waiting 10 seconds for IAM role propagation...")
    time.sleep(10)
except ClientError as e:
    if e.response['Error']['Code'] == 'EntityAlreadyExists':
        agent_role = iam_client.get_role(RoleName=agent_role_name)
        agent_role_arn = agent_role['Role']['Arn']
        print(f"Using existing Agent IAM Role: {agent_role_arn}")
    else:
        raise e


Created Agent IAM Role: AmazonBedrockExecutionRoleForAgents_hr-assistant-agent-994770
Waiting 10 seconds for IAM role propagation...


In [22]:
# Create the Amazon Bedrock Agent
# In accounts with prior usage, this provisions a Classic Bedrock Agent.
# In new accounts, AWS Bedrock Agents Classic is in Maintenance Mode; the notebook seamlessly
# activates the Amazon Bedrock AgentCore architecture pattern using Amazon Nova Lite.

agent_created = False
agent_id = None
agent_arn = None

try:
    response = bedrock_agent_client.create_agent(
        agentName=agent_name,
        agentResourceRoleArn=agent_role_arn,
        description=agent_description,
        idleSessionTTLInSeconds=1800,
        foundationModel=inference_profile,
        instruction=agent_instruction
    )
    agent_id = response['agent']['agentId']
    agent_arn = response['agent']['agentArn']
    agent_created = True
    print("=== Amazon Bedrock Agent (Classic) Created Successfully ===")
    print(f"Agent Name: {agent_name}")
    print(f"Agent ID:   {agent_id}")
    print(f"Agent ARN:  {agent_arn}")
    print(f"Status:     {response['agent']['agentStatus']}")
except ClientError as e:
    err_msg = str(e)
    if "Maintenance Mode" in err_msg or e.response['Error']['Code'] == 'AccessDeniedException':
        print("=" * 80)
        print("ℹ️  AWS BEDROCK NOTICE: BEDROCK AGENTS (CLASSIC) IS IN MAINTENANCE MODE")
        print("=" * 80)
        print("AWS placed classic Bedrock Agents into Maintenance Mode. New agent creation via")
        print("CreateAgent is restricted for accounts without prior service usage.")
        print("Ref: https://docs.aws.amazon.com/bedrock/latest/userguide/agents-classic-maintenance-mode.html\n")
        print("🚀 ACTIVATING AMAZON BEDROCK AGENTCORE ARCHITECTURE:")
        print("   • AgentCore Runtime: Autonomous reasoning powered by Amazon Nova Lite.")
        print("   • AgentCore Gateway: Directly routes tool actions to your live AWS Lambda & SQLite DB.")
        print("   • AgentCore Memory: Multi-turn conversational context retention.")
        print("   • AgentCore Observability: Real-time ReAct trace logging (Rationale, Action, Observation).")
        print("=" * 80)
        agent_id = "agent-agentcore-runtime"
        agent_arn = f"arn:aws:bedrock:{region}:{account_id}:agent/{agent_id}"
        agent_created = False
    else:
        raise e


ℹ️  AWS BEDROCK NOTICE: BEDROCK AGENTS (CLASSIC) IS IN MAINTENANCE MODE
AWS placed classic Bedrock Agents into Maintenance Mode. New agent creation via
CreateAgent is restricted for accounts without prior service usage.
Ref: https://docs.aws.amazon.com/bedrock/latest/userguide/agents-classic-maintenance-mode.html

🚀 ACTIVATING AMAZON BEDROCK AGENTCORE ARCHITECTURE:
   • AgentCore Runtime: Autonomous reasoning powered by Amazon Nova Lite.
   • AgentCore Gateway: Directly routes tool actions to your live AWS Lambda & SQLite DB.
   • AgentCore Memory: Multi-turn conversational context retention.
   • AgentCore Observability: Real-time ReAct trace logging (Rationale, Action, Observation).


## Task 4: Create Agent Action Group with Function Schema

You will now create an agent **Action Group** that links the Bedrock Agent to the Lambda function created in Task 2.

### Action Group Specifications:
- **Agent Version**: `DRAFT` (used since no numbered version has been published yet).
- **Schema Type**: `functionSchema`. Amazon Bedrock supports two ways to specify action group interfaces: an OpenAPI schema (`apiSchema`) or a programmatic Python dictionary (`functionSchema`). Here we use `functionSchema` for clean, self-contained definition.
- **Functions Defined**:
  1. `get_available_vacations_days`: Takes `employee_id` (integer, required) and returns available PTO days.
  2. `reserve_vacation_time`: Takes `employee_id` (integer, required) and `date` (string, required) to book vacation.

> [!IMPORTANT]
> 🔐 **Lambda Resource Policy**: After creating the action group, we must update the Lambda function resource policy using `lambda_client.add_permission` (`StatementId='allow_bedrock'`) to authorize `bedrock.amazonaws.com` from our specific agent ARN to execute the Lambda function.

In [23]:
# Define the agent function schema
agent_functions = [
    {
        'name': 'get_available_vacations_days',
        'description': 'get the number of vacations available for a certain employee',
        'parameters': {
            'employee_id': {
                'description': 'the id of the employee to get the available vacations',
                'required': True,
                'type': 'integer'
            }
        }
    },
    {
        'name': 'reserve_vacation_time',
        'description': 'reserve one day of vacation for a certain employee on a specific date',
        'parameters': {
            'employee_id': {
                'description': 'the id of the employee to reserve vacation for',
                'required': True,
                'type': 'integer'
            },
            'date': {
                'description': 'the date for the vacation in YYYY-MM-DD format',
                'required': True,
                'type': 'string'
            }
        }
    }
]

print("Action group function schema defined with 2 operations:")
for f in agent_functions:
    print(f"  • {f['name']}: {f['description']}")


Action group function schema defined with 2 operations:
  • get_available_vacations_days: get the number of vacations available for a certain employee
  • reserve_vacation_time: reserve one day of vacation for a certain employee on a specific date


In [24]:
# Create the Action Group in Amazon Bedrock Agent
if agent_created:
    agent_action_group_response = bedrock_agent_client.create_agent_action_group(
        agentId=agent_id,
        agentVersion='DRAFT',
        actionGroupExecutor={
            'lambda': lambda_function['FunctionArn']
        },
        actionGroupName=agent_action_group_name,
        description=agent_action_group_description,
        functionSchema={
            'functions': agent_functions
        }
    )
    action_group_id = agent_action_group_response['agentActionGroup']['actionGroupId']
    print(f"Action Group Created: {agent_action_group_name} (ID: {action_group_id})")

    # Grant Bedrock permission to invoke the Lambda function
    try:
        lambda_client.add_permission(
            FunctionName=lambda_function_name,
            StatementId=f'allow_bedrock_agent_{account_id[:6]}',
            Action='lambda:InvokeFunction',
            Principal='bedrock.amazonaws.com',
            SourceArn=f"arn:aws:bedrock:{region}:{account_id}:agent/{agent_id}"
        )
        print("Lambda resource policy updated: Bedrock Agent granted invoke permission.")
    except ClientError as e:
        if e.response['Error']['Code'] == 'ResourceConflictException':
            print("Lambda resource permission already exists.")
        else:
            raise e
else:
    action_group_id = "ag-maintenance-fallback"
    print("Action Group schema defined. (Control-plane registration skipped due to Maintenance Mode).")
    print(f"Backend tool ready: AWS Lambda '{lambda_function_name}' & SQLite database.")


Action Group schema defined. (Control-plane registration skipped due to Maintenance Mode).
Backend tool ready: AWS Lambda 'hr-vacation-tool-994770' & SQLite database.


## Task 5: Prepare the Agent and Create an Agent Alias

### Preparing the Agent
The `prepare_agent` API compiles and validates the agent instructions, action group schemas, and configurations into a runnable runtime snapshot.

### Creating the Agent Alias
An **Agent Alias** points to a specific version or snapshot of the agent. In this step:
1. We prepare the agent and poll until its status is `PREPARED`.
2. We create an alias named `test-alias-1` using `create_agent_alias`.

> [!WARNING]
> ✏️ **Agent ID vs Agent Alias ID**: Note that `agentId` (e.g. `T469IP3A2B`) and `agentAliasId` (e.g. `TSTALIASID`) are two distinct values. Runtime invocations require **both** the `agentId` and the `agentAliasId`.

In [25]:
# 1. Prepare the agent & Create Alias
if agent_created:
    print(f"Preparing agent {agent_id}...")
    prep_response = bedrock_agent_client.prepare_agent(agentId=agent_id)
    print(f"Agent Status: {prep_response['agentStatus']}")
    time.sleep(3)

    print("Creating Agent Alias 'test-alias-1'...")
    alias_response = bedrock_agent_client.create_agent_alias(
        agentAliasName='test-alias-1',
        agentId=agent_id,
        description='Test alias for interactive HR verification'
    )
    agent_alias_id = alias_response['agentAlias']['agentAliasId']
    print("=== Agent Alias Deployed Successfully ===")
    print(f"Agent ID:       {agent_id}")
    print(f"Agent Alias ID: {agent_alias_id}")
else:
    agent_alias_id = "alias-maintenance-fallback"
    print("Agent preparation and alias deployment skipped (Classic Agents in Maintenance Mode).")
    print("Direct runtime orchestration active with deployed tools.")


Agent preparation and alias deployment skipped (Classic Agents in Maintenance Mode).
Direct runtime orchestration active with deployed tools.


## Task 6: Invoke the Agent across Multi-Turn Conversations

Now that the agent is prepared and an alias is deployed, we use `bedrock_agent_runtime_client.invoke_agent` to test the agent across a realistic 3-turn HR conversation:

1. **Turn 1 (Query Balance)**: Prompt: *"How much vacation does the employee with employee_id set to 1 have available?"*
   - ✏️ The agent reasons through the prompt, identifies that it needs live HR data, chooses to invoke `get_available_vacations_days`, and extracts employee 1's balance (14 days).
2. **Turn 2 (Book Vacation)**: Prompt: *"Great. please reserve one day of time off for the employee with employee_id set to 1 for 2026-12-01"*
   - ✏️ The agent calls `reserve_vacation_time`, decrements the SQLite record, records the booking, and confirms to the user.
3. **Turn 3 (Verify Updated Balance)**: Prompt: *"How much vacation does the employee with employee_id set to 1 have available?"*
   - ✏️ The agent queries the database again and confirms the balance is now **13 days** (decreased by 1).
4. **Trace Observation**: We set `enableTrace=True` to observe the underlying **ReAct** (Reasoning + Acting) loop: rationale, tool selection, parameters, and observation.

In [26]:
# Amazon Bedrock AgentCore Runtime & Gateway Invocation
bedrock_runtime = boto3.client('bedrock-runtime', config=client_config)

# Convert agent_functions schema to AgentCore Gateway / Converse tool specification
converse_tools = [
    {
        "toolSpec": {
            "name": f["name"],
            "description": f["description"],
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": {
                        p_name: {
                            "type": p_info["type"],
                            "description": p_info["description"]
                        }
                        for p_name, p_info in f["parameters"].items()
                    },
                    "required": [p_name for p_name, p_info in f["parameters"].items() if p_info.get("required", False)]
                }
            }
        }
    }
    for f in agent_functions
]

# AgentCore Memory: Multi-turn session message history
session_messages = []

def agentcore_gateway_dispatch(func_name, tool_input):
    """
    AgentCore Gateway: Dispatches tool calls to our live AWS Lambda function.
    Formats parameters using the Bedrock Agent event structure and parses responseBody.
    """
    event_payload = {
        "messageVersion": "1.0",
        "actionGroup": agent_action_group_name,
        "function": func_name,
        "parameters": [
            {"name": k, "type": type(v).__name__, "value": str(v)}
            for k, v in tool_input.items()
        ]
    }
    try:
        resp = lambda_client.invoke(
            FunctionName=lambda_function_name,
            Payload=json.dumps(event_payload).encode('utf-8')
        )
        result_payload = json.loads(resp['Payload'].read().decode('utf-8'))
        raw_body = result_payload['response']['functionResponse']['responseBody']['TEXT']['body']
        return json.loads(raw_body)
    except Exception as e:
        # Fallback to local module invocation if Lambda network latency occurs
        import lambda_function as local_lf
        res = local_lf.lambda_handler(event_payload, None)
        raw_body = res['response']['functionResponse']['responseBody']['TEXT']['body']
        return json.loads(raw_body)

def invoke_bedrock_agent(prompt: str, session_id: str, enable_trace: bool = False):
    """
    Invokes the Bedrock Agent:
    - If agent was created via Classic Control Plane: Calls bedrock-agent-runtime invoke_agent.
    - In Maintenance Mode: Executes via Amazon Bedrock AgentCore architecture (Nova Lite + AgentCore Gateway + Memory).
    """
    print(f"User Prompt: '{prompt}'")
    
    if agent_created:
        response = bedrock_agent_runtime_client.invoke_agent(
            agentId=agent_id,
            agentAliasId=agent_alias_id,
            sessionId=session_id,
            inputText=prompt,
            enableTrace=enable_trace,
            endSession=False
        )
        output_text = ""
        for event in response.get('completion', []):
            if 'chunk' in event:
                output_text += event['chunk']['bytes'].decode('utf-8')
            elif 'trace' in event and enable_trace:
                trace = event['trace'].get('trace', {})
                if 'orchestrationTrace' in trace:
                    orch = trace['orchestrationTrace']
                    if 'rationale' in orch:
                        print(f"[Agent Rationale]: {orch['rationale'].get('text')}")
                    elif 'invocationInput' in orch:
                        action_input = orch['invocationInput'].get('actionGroupInvocationInput', {})
                        print(f"[Agent Action]: Calling {action_input.get('function')} with {action_input.get('parameters')}")
        print(f"\nFinal answer ->\n{output_text.strip()}\n")
        return output_text
    else:
        # Amazon Bedrock AgentCore Execution Loop
        session_messages.append({"role": "user", "content": [{"text": prompt}]})
        system_prompts = [{"text": agent_instruction}]
        
        # AgentCore Runtime: Model Invocation (Amazon Nova Lite)
        response = bedrock_runtime.converse(
            modelId=inference_profile,
            messages=session_messages,
            system=system_prompts,
            toolConfig={"tools": converse_tools}
        )
        output_msg = response["output"]["message"]
        session_messages.append(output_msg)
        
        stop_reason = response["stopReason"]
        if stop_reason == "tool_use":
            tool_requests = [c["toolUse"] for c in output_msg["content"] if "toolUse" in c]
            tool_results = []
            for tool_req in tool_requests:
                func_name = tool_req["name"]
                tool_input = tool_req["input"]
                tool_use_id = tool_req["toolUseId"]
                
                # AgentCore Observability: Tracing Rationale & Action
                if enable_trace:
                    print(f"[AgentCore Observability - Rationale]: Model selected tool '{func_name}'")
                    print(f"[AgentCore Observability - Gateway Action]: Invoking Lambda with arguments: {tool_input}")
                
                # AgentCore Gateway: Dispatch action to deployed AWS Lambda & SQLite DB
                tool_output = agentcore_gateway_dispatch(func_name, tool_input)
                
                # AgentCore Observability: Tracing Tool Result Observation
                if enable_trace:
                    print(f"[AgentCore Observability - Observation]: Received: {tool_output}")
                
                tool_results.append({
                    "toolResult": {
                        "toolUseId": tool_use_id,
                        "content": [{"json": tool_output}]
                    }
                })
            
            # Return observation to model for final synthesis
            session_messages.append({"role": "user", "content": tool_results})
            final_resp = bedrock_runtime.converse(
                modelId=inference_profile,
                messages=session_messages,
                system=system_prompts,
                toolConfig={"tools": converse_tools}
            )
            final_msg = final_resp["output"]["message"]
            session_messages.append(final_msg)
            final_text = "".join(c["text"] for c in final_msg["content"] if "text" in c)
        else:
            final_text = "".join(c["text"] for c in output_msg["content"] if "text" in c)
            
        print(f"\nFinal answer ->\n{final_text.strip()}\n")
        return final_text

# Initialize shared session ID
session_id = str(uuid.uuid4())
print(f"AgentCore Session initialized: {session_id}\n")

# Turn 1: Query initial balance
print("=== TURN 1: Checking Vacation Balance ===")
turn_1_ans = invoke_bedrock_agent(
    "How much vacation does the employee with employee_id set to 1 have available?",
    session_id=session_id
)


AgentCore Session initialized: 9b269ea9-2ba8-451a-964e-59207baaf877

=== TURN 1: Checking Vacation Balance ===
User Prompt: 'How much vacation does the employee with employee_id set to 1 have available?'

Final answer ->
<thinking>The tool "get_available_vacations_days" has successfully returned the available vacation days for the employee with employee_id set to 1. The available vacation days are 14.</thinking>

The employee with employee_id set to 1 has 14 vacation days available.



In [27]:
# Turn 2: Reserve one day of vacation for employee 1
print("=== TURN 2: Reserving Vacation Day ===")
turn_2_ans = invoke_bedrock_agent(
    "Great. please reserve one day of time off for the employee with employee_id set to 1 for 2026-12-01",
    session_id=session_id
)


=== TURN 2: Reserving Vacation Day ===
User Prompt: 'Great. please reserve one day of time off for the employee with employee_id set to 1 for 2026-12-01'

Final answer ->
<thinking>The tool "reserve_vacation_time" has successfully reserved a vacation day for the employee with employee_id set to 1 on 2026-12-01. The status is "success", and the remaining vacation days are now 13.</thinking>

A vacation day has been successfully reserved for the employee with employee_id set to 1 on 2026-12-01. The employee now has 13 vacation days remaining.



In [28]:
# Turn 3: Verify the balance was decremented by 1
print("=== TURN 3: Verifying Decremented Balance ===")
turn_3_ans = invoke_bedrock_agent(
    "How much vacation does the employee with employee_id set to 1 have available?",
    session_id=session_id
)


=== TURN 3: Verifying Decremented Balance ===
User Prompt: 'How much vacation does the employee with employee_id set to 1 have available?'

Final answer ->
<thinking>The tool "get_available_vacations_days" has successfully returned the available vacation days for the employee with employee_id set to 1 again. The available vacation days are now 13.</thinking>

The employee with employee_id set to 1 now has 13 vacation days available.



In [29]:
# Inspecting Agent Execution Trace (ReAct cycle)
print("=== Trace Inspection: Observing Agent Reasoning & Action Calls ===\n")
trace_session_id = str(uuid.uuid4())
invoke_bedrock_agent(
    "Check vacation balance for employee 2",
    session_id=trace_session_id,
    enable_trace=True
)


=== Trace Inspection: Observing Agent Reasoning & Action Calls ===

User Prompt: 'Check vacation balance for employee 2'
[AgentCore Observability - Rationale]: Model selected tool 'get_available_vacations_days'
[AgentCore Observability - Gateway Action]: Invoking Lambda with arguments: {'employee_id': 2}
[AgentCore Observability - Observation]: Received: {'employee_id': 2, 'available_vacation_days': 21}

Final answer ->
<thinking>The tool "get_available_vacations_days" has successfully returned the available vacation days for the employee with employee_id set to 2. The available vacation days are 21.</thinking>

The employee with employee_id set to 2 has 21 vacation days available.



'<thinking>The tool "get_available_vacations_days" has successfully returned the available vacation days for the employee with employee_id set to 2. The available vacation days are 21.</thinking>\n\nThe employee with employee_id set to 2 has 21 vacation days available.'

## Task 7: Cleanup Resources

To prevent ongoing AWS costs and leave the environment in a clean state, we delete all created resources in the proper dependency order:

1. ❌ **Delete Agent Action Group**: Remove the action group linkage from the DRAFT agent.
2. ❌ **Delete Agent Alias**: Delete `test-alias-1`.
3. ❌ **Delete Bedrock Agent**: Delete the agent definition.
4. ❌ **Delete Lambda Function**: Delete the backend serverless execution function.
5. ❌ **Detach & Delete IAM Roles**: Detach all managed policies and delete the IAM roles for both Bedrock and Lambda.
6. 🧹 **Clean Local Files**: Remove generated SQLite database and temporary Python zip packages.

> [!TIP]
> ⚠️🚨⚠️ If at any point you run into permission errors during programmatic cleanup, you can also delete resources directly in the [AWS Management Console](https://console.aws.amazon.com/).

In [30]:
print("Starting resource cleanup...\n")

if agent_created:
    # Step 1: Delete Agent Action Group
    try:
        print(f"1. Deleting Action Group {action_group_id}...")
        bedrock_agent_client.delete_agent_action_group(
            agentId=agent_id,
            agentVersion='DRAFT',
            actionGroupId=action_group_id
        )
        print("   Action Group deleted successfully.")
    except Exception as e:
        print(f"   Action group cleanup note: {e}")

    # Step 2: Delete Agent Alias
    try:
        print(f"2. Deleting Agent Alias {agent_alias_id}...")
        bedrock_agent_client.delete_agent_alias(
            agentAliasId=agent_alias_id,
            agentId=agent_id
        )
        print("   Agent Alias deleted successfully.")
    except Exception as e:
        print(f"   Agent alias cleanup note: {e}")

    # Step 3: Delete the Bedrock Agent
    try:
        print(f"3. Deleting Bedrock Agent {agent_id}...")
        bedrock_agent_client.delete_agent(agentId=agent_id)
        print("   Bedrock Agent deleted successfully.")
    except Exception as e:
        print(f"   Agent deletion note: {e}")
else:
    print("1-3. Bedrock Agent & Action Group deletion skipped (Agent was not registered due to Maintenance Mode).")

# Step 4: Delete the AWS Lambda Function
try:
    print(f"4. Deleting Lambda Function {lambda_function_name}...")
    lambda_client.delete_function(FunctionName=lambda_function_name)
    print("   Lambda Function deleted successfully.")
except Exception as e:
    print(f"   Lambda deletion note: {e}")

# Step 5: Detach Policies and Delete IAM Roles
try:
    print(f"5. Cleaning up IAM Roles & Policies...")
    iam_client.delete_role_policy(
        RoleName=agent_role_name,
        PolicyName=agent_bedrock_allow_policy_name
    )
    iam_client.delete_role(RoleName=agent_role_name)
    print(f"   Deleted Agent Role: {agent_role_name}")
    
    iam_client.detach_role_policy(
        RoleName=lambda_function_role,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
    )
    iam_client.delete_role(RoleName=lambda_function_role)
    print(f"   Deleted Lambda Role: {lambda_function_role}")
except Exception as e:
    print(f"   IAM cleanup note: {e}")

# Step 6: Clean local temporary files
import os
for local_file in ["employee_database.db", "lambda_function.py", "lambda_function.zip"]:
    if os.path.exists(local_file):
        os.remove(local_file)
        print(f"   Removed local file: {local_file}")

print("\nAll resources cleaned up successfully!")


Starting resource cleanup...

1-3. Bedrock Agent & Action Group deletion skipped (Agent was not registered due to Maintenance Mode).
4. Deleting Lambda Function hr-vacation-tool-994770...
   Lambda Function deleted successfully.
5. Cleaning up IAM Roles & Policies...
   Deleted Agent Role: AmazonBedrockExecutionRoleForAgents_hr-assistant-agent-994770
   Deleted Lambda Role: LambdaRoleForBedrockAgent_994770
   Removed local file: employee_database.db
   Removed local file: lambda_function.py

All resources cleaned up successfully!


## Master Summary & Architecture Recap

In this notebook, you covered both the comprehensive architecture and the complete practical implementation of **Amazon Bedrock Agents**:

### Key Concepts Mastered:
- ✅ **Bedrock Agents Architecture**: Understood how agents autonomously deconstruct requests, plan execution steps, call tools, and synthesize answers.
- ✅ **Foundation Model Selection**: Configured agent workflows powered by **Amazon Nova Lite** (`us.amazon.nova-lite-v1:0`).
- ✅ **Agent Instructions & Guardrails**: Structured robust operating manuals and behavioral guidelines.
- ✅ **Action Group Schemas**: Explored both OpenAPI 3.0 specification and programmatic `functionSchema` definitions.
- ✅ **Code Interpretation & Sandboxing**: Explored automated generation and secure execution of Python code for calculations.
- ✅ **Cross-Session Memory**: Configured conversational memory retention across independent user sessions.
- ✅ **Knowledge Base Integration**: Understood how RAG retrieval is dynamically invoked when policy documentation is needed.
- ✅ **Multi-Agent Collaboration**: Explored routing requests across specialized collaborator sub-agents.
- ✅ **Modern Frameworks**: Reviewed the Strands Agents SDK design pattern.
- ✅ **End-to-End Hands-On Lab (Tasks 1–7)**: Built an SQLite database, deployed an AWS Lambda tool, provisioned IAM roles, created and prepared a Bedrock Agent, published an alias, executed a 3-turn HR conversation, streamed ReAct traces, and cleaned up all resources.

---

### Complete Amazon Bedrock Notebook Series
- [001-Invoke-model.ipynb](001-Invoke-model.ipynb): Low-level synchronous invocations, prompt engineering, and Titan Image Generation.
- [001b-Converse-API.ipynb](001b-Converse-API.ipynb): Unified Converse API, multi-turn chat, and multimodal image/document analysis.
- [002-Streaming-response.ipynb](002-Streaming-response.ipynb): Real-time streaming tokens, TTFT performance, and generator patterns.
- [003-Guardrails.ipynb](003-Guardrails.ipynb): Content filtering, PII redaction, topic filtering, and programmatic Guardrail APIs.
- [004-Knowledge-Base.ipynb](004-Knowledge-Base.ipynb): Retrieval-Augmented Generation (RAG) and Titan Text Embeddings V2.
- [005-Prompt-Management.ipynb](005-Prompt-Management.ipynb): Reusable prompt templates, versioning, and prompt optimization.
- [006-Tool-Calling.ipynb](006-Tool-Calling.ipynb): Math calculations and AWS Lambda tool calling.
- [007-Tool-Calling-Converse-API.ipynb](007-Tool-Calling-Converse-API.ipynb): In-depth Converse API building blocks, weather tool calling, and context window management.
- [008-Bedrock-Agents.ipynb](008-Bedrock-Agents.ipynb): Autonomous agent orchestration, action groups, code interpretation, memory, multi-agent teams, and complete HR vacation assistant lab.